# 03 — 核心分析

**主要：** Task 7 — 隔年 **wRC+** 預測（過去三季滯後特徵 · Random Forest / XGBoost · 時間切分）  
**輔助：** Task 4–6 — 排名比較、K-Means 分群（逐季 `batting_clean`）、摘要報告


In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'DejaVu Sans']
%matplotlib inline

PROC_DIR = Path('../data/processed')
FIG_DIR  = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 4-1  傳統 vs 進階指標排名比較（找被低估球員）

In [2]:
df = pd.read_csv(PROC_DIR / 'batting_clean.csv')

df['rank_AVG']  = df['AVG'].rank(ascending=False)
df['rank_wRC+'] = df['wRC+'].rank(ascending=False)

# 排名差距：正值 = 被 AVG 低估（wRC+ 排名 比 AVG 排名 更靠前）
df['rank_diff'] = df['rank_AVG'] - df['rank_wRC+']

undervalued = df.nlargest(10, 'rank_diff')[['Name', 'Team', 'AVG', 'rank_AVG', 'wRC+', 'rank_wRC+', 'rank_diff']]
overvalued  = df.nsmallest(10, 'rank_diff')[['Name', 'Team', 'AVG', 'rank_AVG', 'wRC+', 'rank_wRC+', 'rank_diff']]

print('=== 被 AVG 低估的球員（wRC+ 遠高於 AVG 排名）===')
print(undervalued.to_string(index=False))
print('\n=== 被 AVG 高估的球員（wRC+ 遠低於 AVG 排名）===')
print(overvalued.to_string(index=False))

=== 被 AVG 低估的球員（wRC+ 遠高於 AVG 排名）===
            Name Team   AVG  rank_AVG  wRC+  rank_wRC+  rank_diff
Jacob Nottingham  MIL 0.186     899.0 122.7       63.5      835.5
    Eddy Alvarez  MIA 0.194     873.0 118.7       89.0      784.0
      Edwin Ríos  LAD 0.211     789.5 136.8       15.0      774.5
     Luken Baker  STL 0.207     815.0 116.1      113.0      702.0
        Hoy Park  PIT 0.215     745.0 121.4       75.0      670.0
      José Rojas  LAA 0.215     745.0 118.1       93.0      652.0
    Kyle Garlick  MIN 0.213     765.0 116.1      113.0      652.0
  Dustin Garneau  LAA 0.227     625.5 132.1       29.0      596.5
      Joey Gallo  TEX 0.194     873.0 102.7      301.0      572.0
       Greg Bird  NYY 0.209     803.5 106.1      233.5      570.0

=== 被 AVG 高估的球員（wRC+ 遠低於 AVG 排名）===
              Name Team   AVG  rank_AVG  wRC+  rank_wRC+  rank_diff
    Mason Williams  CIN 0.260     200.0  74.7      776.0     -576.0
     Harold Castro  DET 0.279      60.5  85.4      623.0     -562

## 4-2  K-Means 球員分群（4 種打者類型）

In [3]:
df_scaled = pd.read_csv(PROC_DIR / 'batting_scaled.csv')

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(df_scaled)

# Assign labels based on cluster centroids (wRC+ ordering)
cluster_wrc = df.groupby('cluster')['wRC+'].mean().sort_values(ascending=False)
label_order = ['強打型', '選球型', '接觸型', '工具型']
cluster_labels = {cid: label for cid, label in zip(cluster_wrc.index, label_order)}
df['cluster_name'] = df['cluster'].map(cluster_labels)

print('各分群平均指標：')
show_cols = ['AVG', 'OBP', 'SLG', 'BB%', 'K%', 'wRC+']
show_cols = [c for c in show_cols if c in df.columns]
print(df.groupby('cluster_name')[show_cols].mean().round(3))

df.to_csv(PROC_DIR / 'batting_clustered.csv', index=False)
print('\nbatting_clustered.csv 已儲存')

各分群平均指標：
                AVG    OBP    SLG    BB%     K%     wRC+
cluster_name                                            
工具型           0.189  0.256  0.284  0.073  0.298   56.385
強打型           0.260  0.341  0.459  0.100  0.222  117.013
接觸型           0.250  0.310  0.372  0.071  0.197   90.591
選球型           0.226  0.298  0.405  0.083  0.287   93.095

batting_clustered.csv 已儲存


---
## Task 6：分析摘要報告

In [4]:
df = pd.read_csv(PROC_DIR / 'batting_clustered.csv')

print('=' * 50)
print('📊 打者價值分析摘要報告')
print('=' * 50)
print(f'\n資料範圍：2018–2024 Statcast 逐季彙總；此表為 EDA 篩選後之列')
print(f'有效樣本：{len(df)} 名打者（打席數 >= 150）')

print(f'\n--- 關鍵指標相關性 ---')
cols = ['WAR', 'wRC+', 'OBP', 'SLG', 'ISO', 'BB%', 'K%']
cols = [c for c in cols if c in df.columns]
corr_war = df[cols].corr()['WAR'].drop('WAR').sort_values(ascending=False)
for metric, val in corr_war.items():
    print(f'  WAR vs {metric:10s}：{val:+.3f}')

print(f'\n--- 各分群平均 wRC+ ---')
print(df.groupby('cluster_name')['wRC+'].mean().sort_values(ascending=False).round(1))

print(f'\n--- Top 10 打者（依 WAR）---')
show_cols = ['Name', 'Team', 'PA', 'AVG', 'OBP', 'wRC+', 'WAR', 'cluster_name']
show_cols = [c for c in show_cols if c in df.columns]
top10 = df.nlargest(10, 'WAR')[show_cols]
print(top10.to_string(index=False))

print(f'\n--- 被 AVG 最低估的 5 名球員 ---')
df['rank_diff'] = df['AVG'].rank(ascending=False) - df['wRC+'].rank(ascending=False)
undervalued = df.nlargest(5, 'rank_diff')[['Name', 'AVG', 'wRC+', 'rank_diff']]
print(undervalued.to_string(index=False))

📊 打者價值分析摘要報告

資料範圍：2018–2024 MLB 球季（多年彙總）
有效樣本：958 名打者（打席數 >= 150）

--- 關鍵指標相關性 ---
  WAR vs SLG       ：+0.670
  WAR vs wRC+      ：+0.649
  WAR vs OBP       ：+0.625
  WAR vs ISO       ：+0.563
  WAR vs BB%       ：+0.343
  WAR vs K%        ：-0.276

--- 各分群平均 wRC+ ---
cluster_name
強打型    117.0
選球型     93.1
接觸型     90.6
工具型     56.4
Name: wRC+, dtype: float64

--- Top 10 打者（依 WAR）---
            Name Team   PA   AVG   OBP  wRC+  WAR cluster_name
     Aaron Judge  NYY 3710 0.291 0.406 172.1 46.3          強打型
 Freddie Freeman  ATL 4607 0.308 0.396 149.4 44.3          強打型
       Juan Soto  WSH 4205 0.282 0.418 155.4 43.6          強打型
    Bryce Harper  PHI 3936 0.284 0.396 151.4 38.8          強打型
    Mookie Betts  LAD 4122 0.292 0.381 144.8 37.2          強打型
   Shohei Ohtani  LAA 3624 0.282 0.372 152.8 36.4          強打型
      Mike Trout  LAA 2639 0.287 0.410 172.8 33.2          強打型
    José Ramírez  CLE 4302 0.273 0.354 134.1 33.0          強打型
Paul Goldschmidt  STL 4388 0.280 0.363 132.8 32.9 

---
## Task 7：隔年 wRC+ 預測（Player Career Projection）

- **目標：** 球員在 **target_season** 的 **wRC+**（由 `data/processed/projection_panel.parquet` 提供）。
- **特徵：** 前三年 **S−3, S−2, S−1** 的 `K%`、`BB%`、`BABIP`、`BIP%`（共 12 欄）＋各季 PA、三年 PA 合計、`age_t`（S−1 季 `age_bat` 中位數）、`primary_pos`（Lahman / Chadwick；無則 `UNK`）。
- **驗證：** Train 目標年 {2021,2022} · Val **2023** · Test **2024**（不按球員隨機切分，避免未來洩漏）。
- **Walk-forward：** 逐年擴張訓練窗評估 RF。
- **XGBoost：** 若 `import xgboost` 失敗（常見於 macOS 缺 **OpenMP / libomp**），請 `brew install libomp` 後重試；本筆電可能僅跑 RF。


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(ROOT))

from bb_pipeline.train_projection import fit_by_target_season, walk_forward_metrics

PROC_DIR = Path('../data/processed')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_parquet(PROC_DIR / 'projection_panel.parquet')
print('projection_panel:', panel.shape)

results = fit_by_target_season(panel)
print('\n=== Random Forest ===')
print('  Val ', results['rf_val'])
print('  Test', results['rf_test'])

if 'xgb_test' in results:
    print('\n=== XGBoost ===')
    print('  Val ', results['xgb_val'])
    print('  Test', results['xgb_test'])
else:
    print('\n(XGBoost not run — see notebook markdown / install libomp on macOS)')

wf = walk_forward_metrics(panel)
print('\n=== Walk-forward RF ===')
print(wf.to_string(index=False))

imp = pd.Series(results['rf_perm_importance_mean']).sort_values(ascending=False).head(18)
fig, ax = plt.subplots(figsize=(8, 6))
imp.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_title('RF permutation importance (validation season = 2023)')
plt.tight_layout()
out_png = FIG_DIR / 'projection_rf_perm_importance.png'
plt.savefig(out_png, dpi=150)
print('\nSaved', out_png)
plt.show()


In [ ]:
# Predicted vs actual on held-out 2024 (RF)
from sklearn.metrics import mean_absolute_error

rf = results['rf_model']
tr = panel[panel['target_season'].isin((2021, 2022))]
va = panel[panel['target_season'] == 2023]
te = panel[panel['target_season'] == 2024]

from bb_pipeline.train_projection import prepare_xy

x_te, y_te = prepare_xy(te)
pred = rf.predict(x_te)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_te, pred, alpha=0.35, edgecolors='none')
mae_te = mean_absolute_error(y_te, pred)
ax.plot([40, 180], [40, 180], 'k--', lw=1, label='y=x')
ax.set_xlabel('Actual wRC+ (2024)')
ax.set_ylabel('Predicted wRC+ (RF)')
ax.set_title(f'Hold-out 2024 (n={len(te)})  MAE≈{mae_te:.1f}')
ax.legend()
plt.tight_layout()
p2 = FIG_DIR / 'projection_rf_actual_vs_pred_2024.png'
plt.savefig(p2, dpi=150)
print('Saved', p2)
plt.show()
